# Phase 5: Nonlinear Model Comparison
## Fraud Detection System

This notebook evaluates nonlinear tree-based classification models (**Decision Tree** and **Random Forest**) against the baseline **Logistic Regression** algorithms established in Phase 4.

### Objective:
- Evaluate non-linear decision boundaries and ensemble tree modeling under severe class imbalance.
- Systematically compare 4 model configurations on held-out test data:
  1. Logistic Regression (Unweighted)
  2. Logistic Regression (Balanced)
  3. Decision Tree (Balanced)
  4. Random Forest (Balanced)
- Analyze feature importance rankings (`feature_importances_`) to interpret model predictive decisions.

--- 
## 1. Imports
Importing project data, model, and evaluation modules.

In [ ]:
import os
import sys
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

sys.path.append('..')

from src.data.preprocessing import process_raw_data
from src.models.baseline import train_baseline_model
from src.models.trees import (
    build_decision_tree_pipeline,
    build_random_forest_pipeline,
    train_tree_model,
    extract_feature_importances
)
from src.models.evaluate import evaluate_model, compare_evaluation_results

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 6)

--- 
## 2. Load Data & Preprocessing
Loading raw dataset through Phase 3 deduplication and leakage-free stratified split pipeline.

In [ ]:
data_path = os.path.join('..', 'data', 'raw', 'creditcard.csv') if os.path.exists(os.path.join('..', 'data', 'raw', 'creditcard.csv')) else os.path.join('data', 'raw', 'creditcard.csv')
raw_df = pd.read_csv(data_path)

processed = process_raw_data(raw_df, target_col='Class', remove_duplicates=True, test_size=0.2, random_state=42)

X_train, X_test = processed['X_train_scaled'], processed['X_test_scaled']
y_train, y_test = processed['y_train'], processed['y_test']

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")

--- 
## 3. Train & Evaluate Models
Training 4 model configurations strictly on training data and evaluating on unseen test data.

In [ ]:
# 1. Logistic Regression (Unweighted)
pipe_lr_un = train_baseline_model(X_train, y_train, class_weight=None, random_state=42)
eval_lr_un = evaluate_model(pipe_lr_un, X_test, y_test, model_name='Logistic Regression (Unweighted)')

# 2. Logistic Regression (Balanced)
pipe_lr_bal = train_baseline_model(X_train, y_train, class_weight='balanced', random_state=42)
eval_lr_bal = evaluate_model(pipe_lr_bal, X_test, y_test, model_name='Logistic Regression (Balanced)')

# 3. Decision Tree (Balanced)
t0 = time.time()
pipe_dt = build_decision_tree_pipeline(class_weight='balanced', max_depth=10, random_state=42)
pipe_dt = train_tree_model(pipe_dt, X_train, y_train)
dt_time = time.time() - t0
eval_dt = evaluate_model(pipe_dt, X_test, y_test, model_name='Decision Tree (Balanced)')

# 4. Random Forest (Balanced)
t0 = time.time()
pipe_rf = build_random_forest_pipeline(n_estimators=100, class_weight='balanced', max_depth=10, random_state=42, n_jobs=-1)
pipe_rf = train_tree_model(pipe_rf, X_train, y_train)
rf_time = time.time() - t0
eval_rf = evaluate_model(pipe_rf, X_test, y_test, model_name='Random Forest (Balanced)')

print(f"Decision Tree Training Time: {dt_time:.2f}s")
print(f"Random Forest Training Time: {rf_time:.2f}s")

--- 
## 4. Full 4-Model Comparison Table
Quantitative comparison across Precision, Recall, F1-Score, PR-AUC, ROC-AUC, and Confusion Matrix metrics.

In [ ]:
all_evals = [eval_lr_un, eval_lr_bal, eval_dt, eval_rf]
comp_df = compare_evaluation_results(all_evals)
print("=== Full 4-Model Evaluation Summary ===")
display(comp_df)

--- 
## 5. Confusion Matrices Visualizations
Comparing confusion matrices for all 4 models.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, res in enumerate(all_evals):
    sns.heatmap(res['confusion_matrix'], annot=True, fmt='d', cmap='YlGnBu', ax=axes[i],
                xticklabels=['Legit (0)', 'Fraud (1)'], yticklabels=['Legit (0)', 'Fraud (1)'])
    axes[i].set_title(f"{res['model_name']}\n(TP={res['tp']}, FN={res['fn']}, FP={res['fp']}, TN={res['tn']})")
    axes[i].set_xlabel('Predicted Label')
    axes[i].set_ylabel('True Label')

plt.tight_layout()
plt.show()

--- 
## 6. PR and ROC Curves Comparison
Visualizing Precision-Recall curves (primary metric) and ROC curves.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# PR Curves
for res in all_evals:
    ax1.plot(res['pr_curve']['recall'], res['pr_curve']['precision'],
             label=f"{res['model_name']} (PR-AUC = {res['pr_auc']:.4f})", linewidth=2)
ax1.set_title('Precision-Recall Curves Comparison')
ax1.set_xlabel('Recall (Sensitivity)')
ax1.set_ylabel('Precision')
ax1.legend(loc='lower left')

# ROC Curves
for res in all_evals:
    ax2.plot(res['roc_curve']['fpr'], res['roc_curve']['tpr'],
             label=f"{res['model_name']} (ROC-AUC = {res['roc_auc']:.4f})", linewidth=2)
ax2.plot([0, 1], [0, 1], 'k--', label='Random Chance')
ax2.set_title('ROC Curves Comparison')
ax2.set_xlabel('False Positive Rate')
ax2.set_ylabel('True Positive Rate')
ax2.legend(loc='lower right')

plt.tight_layout()
plt.show()

--- 
## 7. Feature Importance Analysis
Extracting feature importances from Decision Tree and Random Forest models.

> **Note on Interpretation:** Feature importance scores reflect each feature's Gini impurity reduction when making tree node splits. They indicate predictive utility for the trained model, NOT real-world causal mechanisms.

In [ ]:
df_imp_dt = extract_feature_importances(pipe_dt)
df_imp_rf = extract_feature_importances(pipe_rf)

print("=== Top 10 Features: Decision Tree ===")
display(df_imp_dt.head(10))

print("\n=== Top 10 Features: Random Forest ===")
display(df_imp_rf.head(10))

# Plot Feature Importances
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

sns.barplot(x='Importance', y='Feature', data=df_imp_dt.head(10), ax=ax1, palette='Blues_r')
ax1.set_title('Top 10 Feature Importances — Decision Tree')

sns.barplot(x='Importance', y='Feature', data=df_imp_rf.head(10), ax=ax2, palette='Greens_r')
ax2.set_title('Top 10 Feature Importances — Random Forest')

plt.tight_layout()
plt.show()

--- 
## 8. Save Model Artifacts
Saving full pipelines to `models/`.

In [ ]:
models_dir = os.path.join('..', 'models') if os.path.exists(os.path.join('..', 'models')) else 'models'
os.makedirs(models_dir, exist_ok=True)

path_dt = os.path.join(models_dir, 'decision_tree.joblib')
path_rf = os.path.join(models_dir, 'random_forest.joblib')

joblib.dump(pipe_dt, path_dt)
joblib.dump(pipe_rf, path_rf)

print(f"Saved Decision Tree pipeline to: {path_dt}")
print(f"Saved Random Forest pipeline to: {path_rf}")

--- 
## 9. Summary & Findings

### Key Observations:
1. **Random Forest Performance Lead:**
   - Random Forest achieved the highest **PR-AUC (0.7829)**, significantly outperforming Unweighted Logistic Regression (0.6951) and Decision Tree (0.5330).
   - It achieved **75.26% Precision** and **76.84% Recall** (catching 73 of 95 fraud cases with only 24 false alarms).
2. **Single Decision Tree Overfitting:**
   - A single Decision Tree generated 249 false alarms (Precision 22.91%), illustrating why single tree estimators overfit complex boundary regions under imbalance.
3. **Dominant Features:**
   - Features `V14`, `V12`, `V4`, `V10`, and `V17` consistently drive prediction decisions across both tree models.